In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pickle

In [2]:
## Load the dataset
data = pd.read_csv("./Churn_Modelling.csv")
print(data.shape)
print(data.head())
print(data.columns.tolist())

(10000, 14)
   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       0  
4

In [3]:
## preprocess the data
## drop the irrelevant variables
## RowNumber, CustomerId, Surname are irrelevant
data = data.drop(columns = ["RowNumber","CustomerId","Surname"], axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
## unique categorical values
print(data["Gender"].unique())
print(data["Geography"].unique())


['Female' 'Male']
['France' 'Spain' 'Germany']


In [5]:
## convert categorical values into numerical values
label_encoder_gender = LabelEncoder()
data["Gender"] = label_encoder_gender.fit_transform(data["Gender"])
print(data["Gender"].unique())

[0 1]


In [6]:
## one hot encoding the geography column
## Frace, Spain, Germany
## any numerical value will weight one more than the others
## better to convert into OHE
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo = OneHotEncoder()
data_geo = onehot_encoder_geo.fit_transform(data[["Geography"]])
data_geo ## we will get a sparse matrix with each row of size 3


<10000x3 sparse matrix of type '<class 'numpy.float64'>'
	with 10000 stored elements in Compressed Sparse Row format>

In [7]:
## obtain the new column names
onehot_encoder_geo.get_feature_names_out(["Geography"])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [8]:
data_geo.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [9]:
## drop the exisiting geography column
data = data.drop(columns=["Geography"],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,0,42,2,0.00,1,1,1,101348.88,1
1,608,0,41,1,83807.86,1,0,1,112542.58,0
2,502,0,42,8,159660.80,3,1,0,113931.57,1
3,699,0,39,1,0.00,2,0,0,93826.63,0
4,850,0,43,2,125510.82,1,1,1,79084.10,0


In [10]:
## conver the data_geo array to pd dataframe before appending
## the column names are same as the array given by feature names
column_names = onehot_encoder_geo.get_feature_names_out(["Geography"])
data_geo_df = pd.DataFrame(data_geo.toarray(),columns=column_names)
data = pd.concat([data,data_geo_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [11]:
## save the scalers and labeles
with open("label_encoder_gender.pkl","wb") as file:
    pickle.dump("label_encoder_gender",file)

with open("onehot_encoder_geo.pkl","wb") as file:
    pickle.dump("onehot_encoder_geo",file)

In [12]:
## divide the dataset into dependent and independent features
X=data.drop("Exited",axis=1) ## independent
y=data["Exited"] ## dependent

## train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,shuffle=True)


## scale the data using Scaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## never fit_transform() the test data
## otherwise it would be similar to leaking the answers


In [13]:
## save the scaled data
with open("scaler.pkl","wb") as file:
    pickle.dump("scaler",file)

In [14]:
## ann implementation
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [15]:
## create an ANN Model using Dense Library
model = Sequential([
    ## create the first hidden layer
    ## specify the shape of the input dataset in the first layer
    ## specify the dimensionality as well by passing it as a tuple
    Dense(64, activation="relu", input_shape = (X_train.shape[1],)),
    ## second hidden layer
    Dense(32, activation="relu"),
    ## output layer
    ## use sigmoid activation function as binary classification is performed
    ## if multiclass classification use softmax
    Dense(1,activation="sigmoid")
])

C:\Users\ashis\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
## reveals all the information regarding the model
## layers, parameters, shape
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
## use optimisers and loss functions and metrics
## useful for model evaluation and weight updation during backpropagation
## insatiate an optimiser with a customised learning rate
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
loss_fn = tf.keras.losses.BinaryCrossentropy()
model.compile(optimizer=opt, loss=loss_fn, metrics=["accuracy"])

In [31]:
print(model.optimizer)
print(model.loss)
print(model.metrics_names)

<LossFunctionWrapper(<function binary_crossentropy at 0x000001BA1F04E2A0>, kwargs={'from_logits': False, 'label_smoothing': 0.0, 'axis': -1})>
['loss', 'compile_metrics']


In [32]:
## set up the tensorboard for visualisation
log_dir = "logs/fit"+datetime.datetime.now().strftime("%Y/%m/%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [33]:
## EarlyStopping ensures efficiency while model training
earlystopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [36]:
## train the model
history = model.fit(
    X_train,
    y_train,
    validation_data = (X_test,y_test),
    epochs = 100,
    callbacks = [tensorflow_callback,earlystopping_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7874 - loss: 0.4526 - val_accuracy: 0.8585 - val_loss: 0.3536
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8533 - loss: 0.3562 - val_accuracy: 0.8610 - val_loss: 0.3477
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8577 - loss: 0.3439 - val_accuracy: 0.8575 - val_loss: 0.3520
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8642 - loss: 0.3319 - val_accuracy: 0.8605 - val_loss: 0.3426
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8630 - loss: 0.3374 - val_accuracy: 0.8590 - val_loss: 0.3474
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8597 - loss: 0.3383 - val_accuracy: 0.8605 - val_loss: 0.3507
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8633 - loss: 0.3346 - val_accuracy: 0.8540 - val_loss: 0.3407
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8612 - loss: 0.3449 - val_accu

In [37]:
model.save('model.h5')

In [48]:
from tensorboard import program

# Set the path to your logs directory (adjust if needed)
log_dir = "logs/fit2025"

# Create and launch the TensorBoard server
tb = program.TensorBoard()
tb.configure(argv=[None, "--logdir", log_dir])
url = tb.launch()

print(f"✅ TensorBoard is running at: {url}")


✅ TensorBoard is running at: http://localhost:6006/


In [ ]:
## Load the pickle file
with open()